# Project implort

In [98]:
import os
import json
import requests
from bs4 import BeautifulSoup

# Ceneo scraper

1. provide URL address of product's opinions webpage

In [99]:
product_code = "174130671"
page = 1
url = f"https://www.ceneo.pl/{product_code}/opinie-{page}"

2. Send a request to provided URL address

In [100]:
response = requests.get(url)
print(response.status_code)

200


3. If status code is OK, fetch product name

In [101]:
page_dom = BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [102]:
product_name = page_dom.find("h1", {"class": "product-top__product-info__name"}).get_text()
print(product_name)

Doppelherz Minoxidil Dla Mężczyzn 60g


4. If status code is OK, fetch all opinions from requested webpage

In [103]:
opinions = page_dom.select("div.js_product-review:not(.user-post-highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
11


In [104]:
opinions = [r for r in page_dom.find_all("div", {"class": "js_product-review"}) if "user-post-highlight" not in r.get("class", [ ])]


5. For all fetched opinions, parse them to extract relevant data

In [105]:
all_opinions = []
for opinion in opinions:
    single_opinion = {
        "opinion_id": opinion["data-entry-id"],
        "author": opinion.select_one("span.user-post__author-name").get_text().strip() if opinion.select_one("span.user-post__author-name") else None,
        "recommendation": opinion.select_one("span.user-post__author-recomendation > em").get_text().strip() if opinion.select_one("span.user-post__author-recomendation > em") else None,
        "score": opinion.select_one("span.user-post__score-count").get_text().strip() if opinion.select_one("span.user-post__score-count") else None,
        "content": opinion.select_one("div.review-feature__item--positive").get_text().strip() if opinion.select_one("div.review-feature__item--positive") else None,
        "pros": [p.get_text() for p in opinion.select("div.review-feature__item--positive")],
        "cons": [c.get_text() for c in opinion.select("div.review-feature__item--negative")],
        "like": opinion.select_one("button.vote-yes > span").get_text().strip() if opinion.select_one("button.vote-yes > span") else None,
        "dislike": opinion.select_one("button.vote-no > span").get_text().strip() if opinion.select_one("button.vote-no > span") else None,
        "publish_date": opinion.select_one("user-post__published > time:nth-child(1)[datatime]")["datetime"].strip() if opinion.select_one("user-post__published > time:nth-child(1)[datatime]") else None,
        "purchase_date": opinion.select_one("user-post__published > time:nth-child(2)[datatime]")["datetime"].strip() if opinion.select_one("user-post__published > time:nth-child(2)[datatime]") else None,
    }

    all_opinions.append(single_opinion)

6. Check if there is next page with opinions

In [106]:
next = True if page_dom.select_one("button.pagination__next") else False
if next: page += 1

8. Save obtained opinions

In [107]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [108]:
with open(f"./opinions/{product_code}.json", "w", encoding="UTF-8") as jf:
    json.dump(all_opinions, jf, indent=4, ensure_ascii=False)